In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from analysis_scripts.graphs import set_size
from analysis_scripts.load_lammps import load_lammps

from pizza.dump import dump

plt.style.use('seaborn')
plt.style.use('tex')
dumpsdir = "/Users/s2469797/Documents/smc-lammps/dumps/"

In [ ]:
testname = "test1_prob1_rate1000_cutoff1.5"
testype = "parallel"

bonds_d = dump(dumpsdir + f"{testname}_bond_{testype}.lammpstrj")
coord_d = dump(dumpsdir + f"{testname}_dump_{testype}.lammpstrj")
try:
    coord_d.unwrap()
except Exception:
    print("Already unwrapped")


In [ ]:
bonds = load_lammps(bonds_d,1001,["c_1[1]","c_1[2]","c_1[3]"],True)
itype = load_lammps(coord_d,1000,["id","type"],True)

In [ ]:
itype2 = itype[(itype[:,:,2]==2)]
bonds2 = bonds[(bonds[:,:,3]==2)]

typedf = pd.DataFrame(itype2, columns=["time","id","type"])
bondsdf = pd.DataFrame(bonds2, columns=["time","bead1", "bead2","type"])

if np.any(bondsdf[["time","type"]].groupby("time").count()>2):
    print("Detected multiple bond")
if np.any(typedf[["time","type"]].groupby("time").count()>2):
    print("Detected multiple type assegnation")

In [ ]:
plt.figure(figsize=set_size(500))
ax = plt.gca()
ax2 = ax.twinx()
ax2.grid(False)

ax.set_xlabel("Time [It]")
ax.set_ylabel("Indexes")

ax.fill_between(bonds2[:,0],bonds2[:,1],bonds2[:,2], alpha =0.7)
ax.scatter(itype2[:,0],itype2[:,1], s=5)


plt.savefig(f"results/kymograph_{testname}_{testype}.pdf")